# 02 — Leakage-safe feature engineering

Features are calculated chronologically. For a transaction, customer and network state contains only earlier events.

In [ ]:
import sqlite3
import pandas as pd
from pipeline.features.engineering import engineer_features

conn = sqlite3.connect('../data/generated/fraud_command_center.db')
tables = {name: pd.read_sql(f'SELECT * FROM {name}', conn) for name in ['transactions','customers','accounts','devices','device_customer_links','beneficiaries']}
features = engineer_features(**tables)
features.head()

## Behavioral and network checks

In [ ]:
features[['amount_vs_customer_average','amount_z_score','transactions_last_1_hour','balance_depletion_ratio']].describe()
features.groupby('customer_segment')['transaction_id'].count()
features[['is_new_device','is_new_beneficiary','is_high_risk_country']].mean()

## Conclusions

The feature table can be audited row by row. Current-row labels such as `is_fraud` are not read by the engineering function, which protects the model evaluation from target leakage.